# 05 — Tiny CNN overfit test

This notebook proves that manifest rows, NetCDF tensors,
labels, normalization, gradients, and a small CNN are wired
together correctly.

It uses exactly 64 frames from the internal 2013 training
split: 32 positive and 32 negative. No validation or
official-test rows are used.

The normalization values calculated here are smoke-test
values only and must not be reused for the real baseline.


In [2]:
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


In [3]:
from pathlib import Path

PACKAGE_VERSION = "0.1.6"
BACKUP_ROOT = Path(
    "/content/drive/MyDrive/TorNet_Backup"
)
PACKAGE_PATH = (
    BACKUP_ROOT
    / "packages"
    / (
        "tornet_detection-"
        f"{PACKAGE_VERSION}-py3-none-any.whl"
    )
)
MANIFESTS_ROOT = BACKUP_ROOT / "manifests"
DRIVE_ARCHIVE_PATH = (
    BACKUP_ROOT / "tornet_2013.tar.gz"
)

LOCAL_ARCHIVE_PATH = Path(
    "/content/tornet_2013.tar.gz"
)
SAMPLE_ROOT = Path(
    "/content/tornet_tiny_overfit"
)

for required_path in (
    PACKAGE_PATH,
    MANIFESTS_ROOT,
    DRIVE_ARCHIVE_PATH,
):
    if not required_path.exists():
        raise FileNotFoundError(
            f"Missing required path: "
            f"{required_path}"
        )

print("package:", PACKAGE_PATH)
print("archive:", DRIVE_ARCHIVE_PATH)


package: /content/drive/MyDrive/TorNet_Backup/packages/tornet_detection-0.1.6-py3-none-any.whl
archive: /content/drive/MyDrive/TorNet_Backup/tornet_2013.tar.gz


In [4]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "netCDF4>=1.7",
    ],
    check=True,
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-deps",
        "--force-reinstall",
        str(PACKAGE_PATH),
    ],
    check=True,
)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '--no-deps', '--force-reinstall', '/content/drive/MyDrive/TorNet_Backup/packages/tornet_detection-0.1.6-py3-none-any.whl'], returncode=0)

In [5]:
import random

import numpy as np
import torch

import tornado_detection

if (
    tornado_detection.__version__
    != PACKAGE_VERSION
):
    raise RuntimeError(
        "Unexpected package version: "
        f"{tornado_detection.__version__}"
    )

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. Select a Colab GPU "
        "runtime and restart the notebook."
    )

SEED = 20260913

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda")

print(
    "tornado_detection:",
    tornado_detection.__version__,
)
print("torch:", torch.__version__)
print("device:", device)
print(
    "GPU:",
    torch.cuda.get_device_name(0),
)


tornado_detection: 0.1.6
torch: 2.11.0+cu128
device: cuda
GPU: NVIDIA A100-SXM4-40GB


In [6]:
from tornado_detection.data import (
    assign_model_splits,
    load_canonical_frame_index,
)

frame_index = load_canonical_frame_index(
    MANIFESTS_ROOT
)
assigned = assign_model_splits(
    frame_index,
    validation_fraction=0.20,
    seed=SEED,
)

eligible = assigned.loc[
    assigned["year"].eq(2013)
    & assigned["model_split"].eq(
        "train"
    )
].copy()

positives = (
    eligible.loc[
        eligible["frame_label"].eq(1)
    ]
    .sort_values("frame_id")
    .sample(
        n=32,
        random_state=SEED,
    )
)

negatives = (
    eligible.loc[
        eligible["frame_label"].eq(0)
    ]
    .sort_values("frame_id")
    .sample(
        n=32,
        random_state=SEED,
    )
)

sample_index = (
    __import__("pandas")
    .concat(
        [
            positives,
            negatives,
        ],
        ignore_index=True,
    )
    .sort_values("frame_id")
    .reset_index(drop=True)
)

assert len(sample_index) == 64
assert (
    sample_index["frame_label"]
    .value_counts()
    .to_dict()
    == {
        0: 32,
        1: 32,
    }
)
assert sample_index[
    "model_split"
].eq("train").all()
assert sample_index[
    "split"
].eq("train").all()

print(
    "sample frames:",
    len(sample_index),
)
print(
    "sample files:",
    sample_index[
        "file_id"
    ].nunique(),
)
print(
    sample_index[
        "frame_label"
    ].value_counts()
    .sort_index()
    .to_string()
)


sample frames: 64
sample files: 63
frame_label
0    32
1    32


In [7]:
import shutil
import tarfile
import time

if LOCAL_ARCHIVE_PATH.exists():
    LOCAL_ARCHIVE_PATH.unlink()

if SAMPLE_ROOT.exists():
    shutil.rmtree(SAMPLE_ROOT)

SAMPLE_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)

copy_started = time.perf_counter()

shutil.copyfile(
    DRIVE_ARCHIVE_PATH,
    LOCAL_ARCHIVE_PATH,
)

copy_seconds = (
    time.perf_counter()
    - copy_started
)

required_members = set(
    sample_index[
        "archive_member"
    ].unique()
)
extracted_members = set()

extraction_started = (
    time.perf_counter()
)

with tarfile.open(
    LOCAL_ARCHIVE_PATH,
    mode="r:gz",
) as archive:
    for member in archive:
        if (
            not member.isfile()
            or member.name
            not in required_members
        ):
            continue

        destination = (
            SAMPLE_ROOT
            / member.name
        )
        destination.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        source_file = archive.extractfile(
            member
        )

        if source_file is None:
            raise RuntimeError(
                f"Could not extract "
                f"{member.name}"
            )

        with (
            source_file,
            destination.open("wb")
            as output_file,
        ):
            shutil.copyfileobj(
                source_file,
                output_file,
                length=1024 * 1024,
            )

        extracted_members.add(
            member.name
        )

extraction_seconds = (
    time.perf_counter()
    - extraction_started
)

missing_members = (
    required_members
    - extracted_members
)

if missing_members:
    raise RuntimeError(
        "Missing sample members: "
        f"{sorted(missing_members)[:10]}"
    )

print(
    "copy seconds:",
    round(copy_seconds, 3),
)
print(
    "required files:",
    len(required_members),
)
print(
    "extraction seconds:",
    round(extraction_seconds, 3),
)


copy seconds: 32.082
required files: 63
extraction seconds: 9.302


In [8]:
import xarray as xr

from tornado_detection.data import (
    build_frame_tensor,
)

tensors = []
labels = []

loading_started = time.perf_counter()

for row in sample_index.itertuples(
    index=False
):
    path = (
        SAMPLE_ROOT
        / row.archive_member
    )

    with xr.open_dataset(
        path,
        engine="netcdf4",
    ) as dataset:
        result = build_frame_tensor(
            dataset,
            int(row.frame_index),
        )

    if (
        result.label
        != int(row.frame_label)
    ):
        raise AssertionError(
            "Manifest/NetCDF label mismatch "
            f"for {row.frame_id}"
        )

    tensors.append(result.values)
    labels.append(result.label)

raw_x = np.stack(
    tensors,
    axis=0,
).astype(
    np.float32,
    copy=False,
)
y = np.asarray(
    labels,
    dtype=np.float32,
)

loading_seconds = (
    time.perf_counter()
    - loading_started
)

assert raw_x.shape == (
    64,
    120,
    240,
    4,
)
assert y.shape == (64,)
assert int(y.sum()) == 32

print("raw tensor shape:", raw_x.shape)
print("labels shape:", y.shape)
print(
    "loading seconds:",
    round(loading_seconds, 3),
)


raw tensor shape: (64, 120, 240, 4)
labels shape: (64,)
loading seconds: 1.857


In [9]:
channel_means = np.nanmean(
    raw_x,
    axis=(
        0,
        1,
        2,
    ),
    dtype=np.float64,
).astype(np.float32)

channel_stds = np.nanstd(
    raw_x,
    axis=(
        0,
        1,
        2,
    ),
    dtype=np.float64,
).astype(np.float32)

if not np.isfinite(
    channel_means
).all():
    raise AssertionError(
        "Non-finite channel means"
    )

if (
    not np.isfinite(
        channel_stds
    ).all()
    or np.any(channel_stds <= 0)
):
    raise AssertionError(
        "Invalid channel standard deviations"
    )

normalized_x = (
    raw_x
    - channel_means.reshape(
        1,
        1,
        1,
        4,
    )
) / channel_stds.reshape(
    1,
    1,
    1,
    4,
)

normalized_x = np.nan_to_num(
    normalized_x,
    nan=0.0,
    posinf=0.0,
    neginf=0.0,
).astype(
    np.float32,
    copy=False,
)

if not np.isfinite(
    normalized_x
).all():
    raise AssertionError(
        "Normalized tensors are not finite"
    )

print(
    "smoke channel means:",
    channel_means.tolist(),
)
print(
    "smoke channel stds:",
    channel_stds.tolist(),
)
print(
    "normalized finite:",
    bool(
        np.isfinite(
            normalized_x
        ).all()
    ),
)


smoke channel means: [25.772871017456055, 25.617172241210938, -0.6216060519218445, 0.3885034918785095]
smoke channel stds: [15.012531280517578, 14.80727767944336, 18.512277603149414, 19.087600708007812]
normalized finite: True


In [10]:
import torch.nn as nn

x_tensor = torch.from_numpy(
    normalized_x
).permute(
    0,
    3,
    1,
    2,
).contiguous().to(device)

y_tensor = torch.from_numpy(
    y
).reshape(
    -1,
    1,
).to(device)

class TinyRadarCNN(nn.Module):
    def __init__(self) -> None:
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(
                4,
                16,
                kernel_size=3,
                padding=1,
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(
                16,
                32,
                kernel_size=3,
                padding=1,
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                padding=1,
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(
                (4, 8)
            ),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(
                64 * 4 * 8,
                128,
            ),
            nn.ReLU(),
            nn.Linear(
                128,
                1,
            ),
        )

    def forward(
        self,
        inputs: torch.Tensor,
    ) -> torch.Tensor:
        return self.classifier(
            self.features(inputs)
        )

model = TinyRadarCNN().to(device)

parameter_count = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print("input:", tuple(x_tensor.shape))
print("labels:", tuple(y_tensor.shape))
print(
    "parameters:",
    f"{parameter_count:,}",
)


input: (64, 4, 120, 240)
labels: (64, 1)
parameters: 286,129


In [11]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
)
loss_function = (
    nn.BCEWithLogitsLoss()
)

losses = []
accuracies = []
gradient_checks = []

training_started = time.perf_counter()

model.train()

for step in range(1, 1001):
    optimizer.zero_grad(
        set_to_none=True
    )

    logits = model(x_tensor)
    loss = loss_function(
        logits,
        y_tensor,
    )

    if not torch.isfinite(loss):
        raise AssertionError(
            f"Non-finite loss at step {step}"
        )

    loss.backward()

    gradients_finite = all(
        parameter.grad is None
        or torch.isfinite(
            parameter.grad
        ).all().item()
        for parameter in model.parameters()
    )

    if not gradients_finite:
        raise AssertionError(
            "Non-finite gradients at "
            f"step {step}"
        )

    optimizer.step()

    with torch.no_grad():
        predictions = (
            logits.sigmoid() >= 0.5
        ).float()
        accuracy = (
            predictions.eq(y_tensor)
            .float()
            .mean()
            .item()
        )

    losses.append(
        float(loss.item())
    )
    accuracies.append(
        float(accuracy)
    )
    gradient_checks.append(
        gradients_finite
    )

    if (
        step == 1
        or step % 100 == 0
        or (
            accuracy == 1.0
            and loss.item() < 0.01
        )
    ):
        print(
            f"step={step:04d} "
            f"loss={loss.item():.6f} "
            f"accuracy={accuracy:.4f}"
        )

    if (
        accuracy == 1.0
        and loss.item() < 0.01
    ):
        completed_step = step
        break
else:
    completed_step = 1000

training_seconds = (
    time.perf_counter()
    - training_started
)

final_loss = losses[-1]
final_accuracy = accuracies[-1]

assert all(gradient_checks)
assert final_accuracy == 1.0
assert final_loss < 0.01
assert final_loss < losses[0]

print()
print(
    "completed step:",
    completed_step,
)
print(
    "initial loss:",
    round(losses[0], 6),
)
print(
    "final loss:",
    round(final_loss, 6),
)
print(
    "final accuracy:",
    round(final_accuracy, 6),
)
print(
    "training seconds:",
    round(training_seconds, 3),
)
print(
    "all gradients finite:",
    all(gradient_checks),
)


step=0001 loss=0.693798 accuracy=0.5000
step=0049 loss=0.009357 accuracy=1.0000

completed step: 49
initial loss: 0.693798
final loss: 0.009357
final accuracy: 1.0
training seconds: 1.671
all gradients finite: True


In [12]:
shutil.rmtree(SAMPLE_ROOT)
LOCAL_ARCHIVE_PATH.unlink()

assert not SAMPLE_ROOT.exists()
assert not LOCAL_ARCHIVE_PATH.exists()

print(
    "Removed all Colab-local overfit artifacts"
)


Removed all Colab-local overfit artifacts
